# 4. 多层感知机 (MLP) (中)

## 4.6 暂退法 (Dropout)

> 如果说“权重衰减”是通过限制参数的取值范围来防止过拟合，那么“暂退法”就是通过破坏网络结构的稳定性来强迫模型变得更健壮。

### 1. 核心动机：抑制“共谋” (Co-adaptation)

在深度学习中，过拟合不仅表现为参数过多，更表现为神经元之间的**共谋 (Co-adaptation)**。

#### 1.1 数学本质：脆弱的平衡
*   **现象**：在没有约束的情况下，模型为了最小化训练误差，可能会演化出一种“脆弱”的权重组合。例如，神经元 $h_1$ 捕捉了一个噪声特征，而神经元 $h_2$ 通过特定的权重分配恰好抵消了 $h_1$ 引入的误差。
*   **数学解释**：这意味着权重矩阵的各个维度之间存在强烈的**协方差 (Covariance)**。这种复杂的相互依赖仅在训练集上有效，一旦遇到未见的测试数据，这种平衡就会崩溃，导致泛化能力极差。
*   **解决思路**：如果我们在训练时随机“抹除”一部分神经元，迫使每个神经元在失去“搭档”的情况下依然能提取有用的特征。

---

### 2. 数学定义与机制

#### 2.1 随机扰动公式
假设隐藏层的输出向量为 $\mathbf{h}$。Dropout 以概率 $p$ 将其元素置为 0，以 $1-p$ 的概率保留并缩放。
对于每个元素 $h_i$，经过 Dropout 后的 $h_i'$ 为：

$$
h_i' = \begin{cases} 0 & \text{概率为 } p \\ \frac{h_i}{1-p} & \text{概率为 } 1-p \end{cases}
$$

#### 2.2 期望值不变性 (Invariance of Expectation)
Dropout 设计的核心在于**无偏性**。为了确保训练和测试时下一层接收到的信号能量一致，必须执行缩放：
$$E[h_i'] = p \cdot 0 + (1-p) \cdot \frac{h_i}{1-p} = h_i$$
**结论**：Dropout 注入的是**乘性噪声**，但在统计意义上保持了输出的期望值不变。

#### 2.3 隐式正则化证明 (Linear Case)
对于线性模型，可以证明最小化 Dropout 后的期望损失等价于：
$$J(\mathbf{w}) = \text{Loss}_{original} + \underbrace{\frac{p}{2(1-p)} \sum_{i} w_i^2 x_i^2}_{\text{隐式 } L_2 \text{ 正则项}}$$
**直觉**：Dropout 强制让模型惩罚那些过大的权重 $w_i$，因为权重越大，随机丢弃带来的波动（方差）就越剧烈。为了收敛，模型必须选择更平滑、更简单的参数。

---

### 3. 关键状态管理：Train vs. Eval

这是 Dropout 实验中最容易产生 Bug 的地方：

| 模式 | 标志位 | Dropout 行为 | 目的 |
| :--- | :--- | :--- | :--- |
| **训练模式** | `net.train()` | **开启**随机丢弃 | 引入噪声，防止神经元共谋，增强泛化。 |
| **评估模式** | `net.eval()` | **关闭**（Identity） | 使用所有神经元的组合力量进行确定性预测。 |

**工程规范建议**：在执行任何评估逻辑（如计算测试集准确率）之前，**必须**显式调用 `net.eval()`，否则测试结果会因为随机性产生剧烈抖动。

---

### 4. 经验法则与暗知识

1.  **放置位置**：通常放在**激活函数 (ReLU) 之后**。
2.  **丢弃率选择**：
    *   靠近输入层的层：$p$ 较小（如 0.2），保护原始特征。
    *   靠近输出层的深层：$p$ 较大（如 0.5），抑制复杂的抽象共谋。
3.  **学习率调整**：使用 Dropout 后，模型收敛变慢，通常可以适当**调大**学习率（$lr$）或增加训练轮数（$epochs$）。
4.  **与权重衰减配合**：Dropout 限制了神经元间的“配合”，L2 限制了权重的“大小”。两者结合是目前对抗过拟合的标准配置。

---

### 5. 总结
**暂退法 (Dropout)** 通过在训练中制造“信息缺失”，迫使神经网络学习到更具普适性的特征组合。它在数学上等价于一种自适应的正则化，在工程上是提高深度模型鲁棒性的核心工具。


In [10]:
# 暂退法的从零实现。
# 注意：Dropout 只在“训练模式”下生效。
import torch
from torch import Tensor, nn
from typing import Optional

def dropout_layer(X: Tensor, dropout_prob: float) -> Tensor:
    """手动实现 Dropout 层。
    
    Args:
        X: 输入张量。
        dropout_prob: 丢失概率 p (0<= p <= 1)。

    Returns:
        经过随机遮罩并缩放后的张量。
    """
    # 1. 边界情况处理
    if dropout_prob == 1.0:
        return torch.zeros_like(X)
    if dropout_prob == 0.0:
        return X
    
    # 2. 生成遮罩 (Mask)
    # torch.rand 生成 [0, 1) 区间的随机浮点数
    # 判断 > p 得到布尔矩阵
    # 转换成 float 后，保留的位置是 1，丢弃的是 0
    mask: Tensor = (torch.rand_like(X) > dropout_prob).float()

    # 3. 元素相乘并执行期望修正 (Scaling)
    return (mask * X) / (1.0 - dropout_prob)

# 测试函数
X_test = torch.arange(16, dtype=torch.float32).reshape((2, 8))
print(f"原始数据:\n{X_test}")
print(f"p=0.5 时的 Dropout 输出:\n{dropout_layer(X_test, 0.5)}")

原始数据:
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
p=0.5 时的 Dropout 输出:
tensor([[ 0.,  0.,  4.,  0.,  0.,  0., 12.,  0.],
        [16., 18.,  0., 22.,  0., 26., 28., 30.]])


> 这个 Python 注释竟然支持 markdown 语法。

In [11]:
# 带有 Dropout 的多层感知机
# 该实验用的数据集是那个 Fashion-MNIST。
class DropoutMLPScratch(nn.Module):
    """从零实现带 Dropout 的多层感知机 (两个隐藏层)，即简洁实现的 nn.Sequential 容器。
    
    **继承 nn.Module 的关键特性**:
        1. **参数管理** - `nn.Module` 自动追踪所有子模块中的参数 (权重和偏置)。
        2. **训练/评估模式** - 通过 `.trian()` 和 `.eval()` 方法切换，Dropout 会根据模式自动启用或禁用。
        3. **梯度计算** - 自动支持反向传播和梯度更新。
        4. **模型移动** - 支持 `.to(device)` 将整个模型移到 GPU/CPU。
        5. **参数访问** - 可以通过 `.parameters()`、`.named_parameters()` 等方法访问所有参数。
        6. **子模块注册** - 在 `__init__` 中定义的子模块会自动注册。
    """

    def __init__(self, num_inputs: int, num_outputs: int, num_hiddens1: int,
                 num_hiddens2: int, dropout1: float, dropout2: float):
        super().__init__()
        self.num_inputs = num_inputs

        # 定义层参数
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)
        self.relu = nn.ReLU()

        self.dropout1 = dropout1
        self.dropout2 = dropout2

    def forward(self, X: Tensor) -> Tensor:
        """前向传播 (即定义的模型)。"""

        # 1. 展平输入
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))
        
        # 2. 第一层 Dropout: 仅在训练模式下起效。
        if self.training:
            H1 = dropout_layer(H1, self.dropout1)

        H2 = self.relu(self.lin2(H1))
        
        # 3. 第二层 Dropout
        if self.training:
            H2 = dropout_layer(H2, self.dropout2)

        return self.lin3(H2)

In [12]:
# 暂退法的简洁实现
def get_dropout_net(
    num_inputs: int,
    num_outputs: int,
    num_hiddens1: int,
    num_hiddens2: int,  
    dropout1: float,
    dropout2: float 
) -> nn.Sequential:
    """使用 PyTorch 内置层构建带 Dropout 的 MLP。"""

    net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(num_inputs, num_hiddens1),
        nn.ReLU(),
        # 插入 Dropout 层
        nn.Dropout(dropout1),
        nn.Linear(num_hiddens1, num_hiddens2),
        nn.ReLU(),
        nn.Dropout(dropout2),
        nn.Linear(num_hiddens2, num_outputs)
    )

    # 统一初始化
    def init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.01)
            if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    net.apply(init_weights)
    return net

**丢弃率 $p$ 通常设为多少？**
*   通常在 `0.2` 到 `0.5` 之间。
*   **经验**：靠近输入层的层可以设小一点（比如 0.2），靠近输出层的深层可以设大一点（比如 0.5）。

In [13]:
# 实验参数配置
from dataclasses import dataclass

@dataclass(frozen=True)
class MLPConfig:
    num_inputs: int = 784
    num_outputs: int = 10
    num_hiddens1: int = 256
    num_hiddens2: int = 256
    dropout1: float = 0.0       # 默认不开启
    dropout2: float = 0.0
    lr: float = 0.5             # Dropout 训练通常可以用大一点的学习率
    num_epochs: int = 10
    batch_size: int = 256

In [14]:
# 开始实验
import d2l_utils as d2l

# 1. 加载数据集
train_iter, test_iter = d2l.load_fashion_mnist(batch_size=256)
loss_fn = nn.CrossEntropyLoss(reduction='none')

# 2. 定义实验 A: 无 Dropout (过拟合对照组)
config_none = MLPConfig()
net_none = get_dropout_net(
    config_none.num_inputs,
    config_none.num_outputs,
    config_none.num_hiddens1,
    config_none.num_hiddens2,
    config_none.dropout1,
    config_none.dropout2
)
train_none = torch.optim.SGD(net_none.parameters(), lr=config_none.lr)

print(">>> 开始训练：无 Dropout 模型")
d2l.train_softmax(net_none, train_iter, test_iter, loss_fn, config_none.num_epochs, train_none)

# 3. 定义实验 B: 有 Dropout (手写实现组、history_dropout2 =  简化实现组)
config_dropout = MLPConfig(dropout1=0.2, dropout2=0.5)
net_dropout1 = DropoutMLPScratch(
    config_dropout.num_inputs,
    config_dropout.num_outputs,
    config_dropout.num_hiddens1,
    config_dropout.num_hiddens2,
    config_dropout.dropout1,
    config_dropout.dropout2
)
net_dropout2 = get_dropout_net(
    config_dropout.num_inputs,
    config_dropout.num_outputs,
    config_dropout.num_hiddens1,
    config_dropout.num_hiddens2,
    config_dropout.dropout1,
    config_dropout.dropout2
)
train_dropout1 = torch.optim.SGD(net_dropout1.parameters(), config_dropout.lr)
train_dropout2 = torch.optim.SGD(net_dropout2.parameters(), config_dropout.lr)
print("\n>>> 开始训练：带 Dropout 模型 (Scratch)")
d2l.train_softmax(net_dropout1, train_iter, test_iter, loss_fn, config_dropout.num_epochs, train_dropout1)

print("\n>>> 开始训练：带 Dropout 模型 (Concise)")
d2l.train_softmax(net_dropout2, train_iter, test_iter, loss_fn, config_dropout.num_epochs, train_dropout2)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
>>> 开始训练：无 Dropout 模型
开始训练，总轮数: 10。
Epoch 1: Loss = 1.2559, Train Acc = 0.5197, Test Acc = 0.6514
Epoch 2: Loss = 0.5866, Train Acc = 0.7732, Test Acc = 0.8087
Epoch 3: Loss = 0.4710, Train Acc = 0.8227, Test Acc = 0.8145
Epoch 4: Loss = 0.4198, Train Acc = 0.8433, Test Acc = 0.8411
Epoch 5: Loss = 0.3846, Train Acc = 0.8555, Test Acc = 0.8439
Epoch 6: Loss = 0.3628, Train Acc = 0.8633, Test Acc = 0.8179
Epoch 7: Loss = 0.3483, Train Acc = 0.8688, Test Acc = 0.8096
Epoch 8: Loss = 0.3346, Train Acc = 0.8744, Test Acc = 0.8046
Epoch 9: Loss = 0.3208, Train Acc = 0.8786, Test Acc = 0.8578
Epoch 10: Loss = 0.3049, Train Acc = 0.8852, Test Acc = 0.8331
训练完成！

>>> 开始训练：带 Dropout 模型 (Scratch)
开始训练，总轮数: 10。
Epoch 1: Loss = 0.8849, Train Acc = 0.6693, Test Acc = 0.7197
Epoch 2: Loss = 0.5303, Train Acc = 0.8041, Test Acc = 0.7503
Epoch 3: Loss = 0.4683, Train Acc = 0.8278, Test Acc = 0.8309
Epoch 4: Loss = 0.4222, Train A

## 4.7 前向传播、反向传播和计算图

### 1. 前向传播 (Forward Propagation)

#### 1.1 定义
前向传播是指按顺序（从输入层到输出层）计算和存储神经网络中每层变量的过程。

#### 1.2 数学链路（以单隐藏层 MLP 为例）
为了计算损失，模型会依次执行以下动作：
1.  **中间变量**：$\mathbf{z} = \mathbf{W}^{(1)} \mathbf{x}$ （计算线性加权，忽略偏置以简化）
2.  **激活变量**：$\mathbf{h} = \phi(\mathbf{z})$ （应用非线性激活函数）
3.  **输出层变量**：$\mathbf{o} = \mathbf{W}^{(2)} \mathbf{h}$
4.  **损失项**：$L = l(\mathbf{o}, y)$
5.  **正则项**：$s = \frac{\lambda}{2} (\|\mathbf{W}^{(1)}\|_F^2 + \|\mathbf{W}^{(2)}\|_F^2)$
6.  **最终目标函数**：$J = L + s$

---

### 2. 计算图 (Computational Graph)

#### 2.1 什么是计算图？
计算图是将上述数学操作可视化的图形。
*   **节点 (Nodes)**：表示变量（张量）。
*   **边 (Edges)**：表示操作（算子，如矩阵乘法、加法、ReLU）。

#### 2.2 动态图特性 (PyTorch 核心)
PyTorch 使用的是**命令式编程（Imperative Programming）**和**动态计算图**：
*   图是在代码运行时“现场”搭建的。
*   **工程影响**：你可以使用 Python 的 `if` 和 `for` 来改变网络结构，PyTorch 依然能正确求导。

---

### 3. 反向传播 (Backward Propagation)

#### 3.1 定义
反向传播是根据微积分中的**链式法则 (Chain Rule)**，按相反顺序（从输出层到输入层）遍历计算图，计算目标函数关于各个参数梯度的过程。

#### 3.2 链式法则的传递
假设我们要算 $J$ 对 $W^{(1)}$ 的梯度：
$$\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial L} \cdot \frac{\partial L}{\partial \mathbf{o}} \cdot \frac{\partial \mathbf{o}}{\partial \mathbf{h}} \cdot \frac{\partial \mathbf{h}}{\partial \mathbf{z}} \cdot \frac{\partial \mathbf{z}}{\partial \mathbf{W}^{(1)}} + \frac{\partial J}{\partial s} \cdot \frac{\partial s}{\partial \mathbf{W}^{(1)}}$$

---

### 4. 关键工程暗知识：显存管理 (Memory Logic)

这是 4.7 节对实际开发影响最大的地方，请务必掌握：

#### 4.1 为什么训练需要更多显存？
*   **核心逻辑**：在反向传播计算梯度时，我们需要用到前向传播时的**中间结果**（如上述的 $\mathbf{h}$ 和 $\mathbf{z}$）。
*   **后果**：因此，在前向传播过程中，PyTorch **必须保留**所有中间层的激活值，不能丢弃。直到 `backward()` 结束，这些显存才会被释放。
*   **结论**：网络越深、Batch Size 越大，需要存储的中间激活值就越多，越容易导致显存溢出。

#### 4.2 为什么推理（Inference）更省资源？
*   当我们使用 `with torch.no_grad():` 时，PyTorch 知道不需要进行反向传播。
*   **优化**：它会在前向传播算完每一层后，立即丢弃不需要的中间变量。

---


### 5. 模块化代码验证：观察计算图节点

In [2]:
# 观察计算图节点。
import torch
from torch import Tensor

def inspect_computational_graph() -> None:
    """通过打印 grad_fn 观察计算图的节点。"""
    x: Tensor = torch.randn(1, 10, requires_grad=True)
    w: Tensor = torch.randn(10, 5, requires_grad=True)

    # 1. 执行操作
    z = x @ w
    h = torch.relu(z)

    # 2. 观察 grad_fn (指向图中上一个操作的指针)
    print(f"ReLU 层的梯度函数: {h.grad_fn}")

    print(f"MatMul 层的梯度函数: {z.grad_fn}")

    # 3. 根节点(输入)没有 grad_fn
    print(f"输入 x 的梯度函数: {x.grad_fn}")

inspect_computational_graph()

ReLU 层的梯度函数: <ReluBackward0 object at 0x7e023c4b8a90>
MatMul 层的梯度函数: <MmBackward0 object at 0x7e03b8a7f760>
输入 x 的梯度函数: None


---

### 6. 总结：前向与反向的博弈

| 维度 | 前向传播 | 反向传播 |
| :--- | :--- | :--- |
| **方向** | 输入 $\to$ 输出 | 损失 $\to$ 参数 |
| **产出** | 预测值 & 损失值 | 参数的梯度 ($\nabla$) |
| **显存行为** | **存储**激活值 | **消耗**激活值，**产出**梯度 |
| **数学工具** | 代数运算 | 链式法则 |

---

### **“如何减少训练时的显存占用？”**
1.  减小 Batch Size。
2.  使用 `torch.utils.checkpoint`（时间换空间策略：不存中间值，反向传播时临时重算）。
3.  减少网络层数或宽度。


## 4.8 数值稳定性与模型初始化

> 在深度学习中，随着层数的增加，数学运算的细微偏差会被指数级放大，导致训练彻底失败。

### 1. 数值稳定性的数学根源：链式法则的累积效应

假设一个深层网络有 $L$ 层，每一层 $l$ 的输出为 $\mathbf{h}^l = f_l(\mathbf{h}^{l-1})$，最终损失为 $L$。
计算损失关于第一层权重 $\mathbf{W}^1$ 的梯度时，根据链式法则：
$$\frac{\partial L}{\partial \mathbf{W}^1} = \frac{\partial L}{\partial \mathbf{h}^L} \cdot \frac{\partial \mathbf{h}^L}{\partial \mathbf{h}^{L-1}} \cdot \dots \cdot \frac{\partial \mathbf{h}^1}{\partial \mathbf{W}^1}$$

**核心问题**：中间这一连串的矩阵乘积 $\prod_{i=2}^L \frac{\partial \mathbf{h}^i}{\partial \mathbf{h}^{i-1}}$ 是导致不稳定的元凶。
*   如果这些项大多**小于 1**：经过 $L$ 次方后，梯度指数级减小 $\to$ **梯度消失**。
*   如果这些项大多**大于 1**：经过 $L$ 次方后，梯度指数级增大 $\to$ **梯度爆炸**。

#### 1.1 梯度消失 (Vanishing Gradients)
*   **现象**：靠近输入层的参数更新极其缓慢，甚至完全停止。
*   **数学根源**：链式法则中中间项的乘积 $\prod \frac{\partial \mathbf{h}^i}{\partial \mathbf{h}^{i-1}}$ 远小于 1。
*   **典型案例：Sigmoid 激活函数**：
    *   $\sigma(x) = \frac{1}{1 + e^{-x}}$，导数 $\sigma'(x) = \sigma(x)(1 - \sigma(x))$。
    *   **致命伤**：其导数最大值仅为 **0.25**（当 $x=0$ 时）。
    *   **累积效应**：对于 20 层网络，底层梯度会萎缩到 $0.25^{20} \approx 10^{-12}$，导致深层结构无法被有效训练。

#### 1.2 梯度爆炸 (Exploding Gradients)
*   **现象**：损失值变为 `NaN`，权重数值超出浮点数表示范围。
*   **数学根源**：中间项乘积远大于 1，通常由初始权重过大引起。
*   **后果**：模型发散，训练瞬间崩溃。

---

### 2. 隐藏的危机：参数对称性 (Symmetry)

**为什么不能初始化为全 0 或相同的常数？**
*   **数学解释**：如果一层的所有权重 $w_{ij}$ 都相同，那么该层的所有神经元在前向传播时会输出相同的值，在反向传播时会接收到完全相同的梯度。
*   **后果**：无论隐藏层有多少个神经元，它们都在做完全重复的计算。这被称为“对称性问题”，导致增加神经元维度失去了提取不同特征的意义。
*   **结论**：初始化必须是**随机的**，以打破对称性。

---

### 3. 破局关键：方差一致性理论

为了使信息在极深的网络中平稳流动，我们需要遵循以下核心目标：
1.  **前向传播**：每层输出的方差保持一致，防止信号消失。
2.  **反向传播**：每层梯度的方差保持一致，防止梯度消失。

#### 3.1 Xavier 初始化 (Glorot Initialization)
*   **适用场景**：激活函数是线性的，或者类似 **Sigmoid / Tanh**。
*   **数学原理**：为了保持信号强度，对于有 $n_{in}$ 个输入和 $n_{out}$ 个输出的层，权重的方差应满足：
    $$\sigma^2 = \frac{2}{n_{in} + n_{out}}$$
*   **分布形式**：$W \sim \mathcal{N}(0, \sqrt{\frac{2}{n_{in} + n_{out}}})$ 或 $\mathcal{U}(-\sqrt{\frac{6}{n_{in} + n_{out}}}, \sqrt{\frac{6}{n_{in} + n_{out}}})$。
*   **PyTorch 调用**：`nn.init.xavier_uniform_` 或 `nn.init.xavier_normal_`。

#### 3.2 Kaiming 初始化 (He Initialization)
*   **适用场景**：激活函数是 **ReLU**。
*   **数学原理**：由于 ReLU 会随机“杀掉”一半的负信号，因此方差需要翻倍来补偿：
    $$\sigma^2 = \frac{2}{n_{in}}$$
*   **PyTorch 调用**：`nn.init.kaiming_uniform_` 或 `nn.init.kaiming_normal_`。

---

### 3. 关键工程暗知识 (Engineering Wisdom)

1.  **激活函数的演进**：
    *   **ReLU** 及其变体（Leaky ReLU）是目前解决梯度消失最有效的手段，因为它们在正区间的导数恒为 1。
2.  **Batch Normalization (批归一化)**：
    *   这是 4.8 节数学原理的工程终极解决方案（将在第 7 章详讲）。它通过强制每一层输出回到标准正态分布，彻底解决了深层网络的稳定性问题。
3.  **预训练与微调**：
    *   在现代大模型（如 BERT, Llama）中，我们不再从头初始化，而是使用预训练好的权重，这本质上是站在巨人的肩膀上避开了不稳定的初始化阶段。

---

### 4. 总结
数值稳定性是深度学习的“生命线”。
*   **梯度消失**：让网络变“死”，学不动。
*   **梯度爆炸**：让网络变“疯”，计算崩溃。
*   **初始化**：是第一道防线，确保信号能传得远、回得来。

**初始化与激活函数的配对表**：

| 激活函数 | 推荐初始化算法 | PyTorch 函数 | 备注 |
| :--- | :--- | :--- | :--- |
| **ReLU** | Kaiming (He) | `kaiming_normal_` | 深度学习默认组合，最稳健。 |
| **Leaky ReLU** | Kaiming (He) | `kaiming_normal_` | 需指定 `a` 参数（负斜率）。 |
| **Tanh** | Xavier (Glorot) | `xavier_normal_` | 保持输出在 $[-1, 1]$ 之间。 |
| **Sigmoid** | Xavier (Glorot) | `xavier_normal_` | 现已较少在隐藏层使用。 |
| **None (Linear)** | Xavier (Glorot) | `xavier_uniform_` | 仅做线性变换时。 |

In [1]:
# 梯度稳定性演示实验
import torch
from torch import Tensor, nn

def simulate_gradient_flow(num_layers: int, num_inputs: int, std: float) -> Tensor:
    """模拟深层网络中的梯度流，观察梯度消失/爆炸。

    它计算最终输出对第一层权重的梯度，以此衡量数值稳定性。
    
    Args:
        num_layers: 网络的层数。
        num_inputs: 神经元维度。
        std: 初始化权重的标准差。

    Returns:
        Tensor: 第一层权重的梯度张量。
    """
    # 1. 准备输入数据 (均值为 0，方差为 1)
    x: Tensor = torch.randn(1, num_inputs)

    # 2. 初始化第一层权重，并开始梯度追踪
    # 我们重点观察这一层的梯度 w1.grad
    w1: Tensor = torch.normal(0, std, size=(num_inputs, num_inputs), requires_grad=True)
    
    # 3. 执行第一层前向传播
    current_h = x @ w1

    # 4. 模拟深层线性堆叠
    # 为了简化数学本质，这里不加入激活函数，纯粹观察矩阵连乘效应
    for _ in range(num_layers-1):
        w = torch.normal(0, std, size=(num_inputs, num_inputs))
        current_h = current_h @ w

    # 5. 计算损失并反向传播
    loss = current_h.sum()
    loss.backward()
    return w1.grad

# A. 观察梯度爆炸
print(">>> 实验 A: std = 1.0 (高方差初始化)")
grad_exploding = simulate_gradient_flow(num_layers=100, num_inputs=256, std=1.0)
print(f"第一层梯度的平均绝对值: {grad_exploding.abs().mean().item():.2e}")
print(f"是否存在 NaN 或 Inf: {torch.isnan(grad_exploding).any() or torch.isinf(grad_exploding).any()}")

# B. 观察梯度消失
print("\n>>> 实验 B: std = 0.01 (低方差初始化)")
grad_vanishing = simulate_gradient_flow(num_layers=100, num_inputs=256, std=0.01)
print(f"第一层梯度的平均绝对值: {grad_vanishing.abs().mean().item():.2e}")

>>> 实验 A: std = 1.0 (高方差初始化)
第一层梯度的平均绝对值: nan
是否存在 NaN 或 Inf: True

>>> 实验 B: std = 0.01 (低方差初始化)
第一层梯度的平均绝对值: 0.00e+00


In [2]:
# 构建复杂的深层网络
def get_deep_mlp(
    num_inputs: int, 
    num_hiddens: int, 
    num_outputs: int, 
    num_layers: int,
    activation: str = "relu"
) -> nn.Sequential:
    """构建一个极深的多层感知机用于压力测试。"""
    layers: list[nn.Module] = [nn.Flatten()]

    # 映射激活函数字符串到类
    act_layer = nn.ReLU if activation == "relu" else nn.Tanh

    # 输入层
    layers.append(nn.Linear(num_inputs, num_hiddens))
    layers.append(act_layer())

    # 堆叠隐藏层 (共 num_layers - 2 层，因为输入层和输出层各占 1)
    for _ in range(num_layers - 2):
        layers.append(nn.Linear(num_hiddens, num_hiddens))
        layers.append(act_layer())

    # 输出层
    layers.append(nn.Linear(num_hiddens, num_outputs))
    return nn.Sequential(*layers)

| 监控对象 | 监控时机 | 解决的问题 |
| :--- | :--- | :--- |
| **层输出 (Activations)** | 前向传播 | 验证**初始化**、**激活函数**是否合理。 |
| **层梯度 (Gradients)** | 反向传播 | 验证**学习率**、**损失函数缩放**、**残差连接**是否有效。 |

In [7]:
# 数据稳定性验证工具
# 用来监控 "梯度流"
import math

@torch.no_grad()
def check_layer_stats(net: nn.Sequential, num_inputs: int) -> dict[str, float]:
    """监控每一层输出的均值和标准差。
    
    逐层打印并返回最终层的状态。
    """
    net.eval()
    # 生成标准正态分布的伪数据 (1000个样本)
    x: Tensor = torch.randn(1000, num_inputs) # 标准正态分布输入

    print(f"{'Layer':<10} | {'Mean':<15} | {'Std':<15}")
    print("-" * 45)
    
    current_x: Tensor = x
    layer_idx = 0 # 记录层序号
    final_mean: float = 0.0
    final_std: float = 0.0

    for layer in net:
        current_x: Tensor = layer(current_x)
        # 只监控线性层的输出
        if isinstance(layer, nn.Linear):
            layer_idx += 1

            mean, std = current_x.mean().item(), current_x.std().item()
            final_mean, final_std = mean, std

            # 格式化输出: 大于 1e6 的数用科学而计数法
            mean_str = f"{mean:.4e}" if abs(mean) > 1e6 else f"{mean:>10.4f}"
            std_str = f"{std:.4e}" if std > 1e6 else f"{std:>10.4f}"

            print(f"第 {layer_idx:2} 层    | {mean_str:>15} | {std_str:>15}")

            # 如果数值已经爆炸或消失，提前终止
            if not (math.isfinite(mean) and math.isfinite(std)):
                print(">>> 数值已崩溃，停止监控。")
                break
            
            if std < 1e-5:
                print(">>> 梯度消失，停止监控。")
                break
            
    return {"final_mean": final_mean, "final_std": final_std}

In [8]:
# 较为稳定的初始化函数。
from typing import Callable
def apply_advanced_init(nonlinearity: str = "relu") -> Callable[[nn.Module], None]:
    """返回一个针对特定激活函数优化的初始化函数。
    
    根据层类型自动选择最佳初始化策略:
    - Linear + ReLU -> 使用 Kaiming 初始化。
    - Linear + Sigmoid/Tanh -> 使用 Xavier 初始化

    Args:
        nonlinearity: 激活函数名称，如 "relu", "leaky_relu", "tanh", "sigmoid"。

    Returns:
        Callable: 可直接用于 net.apply() 的初始化函数。
    """
    def init_weights(m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            if nonlinearity in ["relu", "leaky_relu"]:
                # Kaiming 初始化 (针对 ReLU 系列优化)
                nn.init.kaiming_normal_(m.weight, nonlinearity=nonlinearity)
            elif nonlinearity in ["tanh", "sigmoid"]:
                # Xavier 初始化 (针对 Tanh/Sigmoid 优化)
                nn.init.xavier_normal_(m.weight)
            else:
                # 默认使用 Xavier 均匀分布
                nn.init.xavier_uniform_(m.weight)

            # 偏置初始化：通常设置为常数 0
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    return init_weights

# 调用示例

# 场景 A: 你的模型使用的是 ReLU
net_relu = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
net_relu.apply(apply_advanced_init(nonlinearity='relu'))

# 场景 B: 你的模型使用的是 Tanh (比如一些循环神经网络)
net_tanh = nn.Sequential(nn.Linear(784, 256), nn.Tanh(), nn.Linear(256, 10))
net_tanh.apply(apply_advanced_init(nonlinearity='tanh'))

Sequential(
  (0): Linear(in_features=784, out_features=256, bias=True)
  (1): Tanh()
  (2): Linear(in_features=256, out_features=10, bias=True)
)

In [9]:
# 对比实验
print("\n=== 场景 1: 朴素初始化 (std=1.0) ===")
net_bad = get_deep_mlp(784, 256, 10, 40)
for m in net_bad:
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=1.0) # 故意设大

check_layer_stats(net_bad, 784)
# 预期结果：Std 会迅速变成 inf，然后 Mean 变成 nan。

print("\n=== 场景 2: Kaiming 初始化 (针对 ReLU) ===")
net_kaiming = get_deep_mlp(784, 256, 10, 40, activation='relu')
net_kaiming.apply(apply_advanced_init(nonlinearity='relu'))

check_layer_stats(net_kaiming, 784)
# 预期结果：即使 40 层，Std 依然保持在 0.6~1.2 之间，非常稳定。

print("\n=== 场景 3: Xavier 初始化 (针对 Tanh) ===")
net_xavier = get_deep_mlp(784, 256, 10, 40, activation='tanh')
net_xavier.apply(apply_advanced_init(nonlinearity='tanh'))

check_layer_stats(net_xavier, 784)
# 预期结果：Mean 极度接近 0，Std 稳定。


=== 场景 1: 朴素初始化 (std=1.0) ===
Layer      | Mean            | Std            
---------------------------------------------
第  1 层    |          0.0766 |         27.9833
第  2 层    |         -9.6425 |        314.4310
第  3 层    |        -98.0739 |       3428.8247
第  4 层    |       -470.8677 |      37424.1367
第  5 层    |      18832.4355 |     405293.1562
第  6 层    |      30935.9160 |      4.6673e+06
第  7 层    |     -5.3694e+06 |      5.5871e+07
第  8 层    |      3.7653e+07 |      5.7340e+08
第  9 层    |      7.4275e+07 |      6.9924e+09
第 10 层    |     -1.3105e+09 |      7.8376e+10
第 11 层    |      2.0806e+10 |      8.3527e+11
第 12 层    |      3.0972e+10 |      8.9854e+12
第 13 层    |      8.6297e+12 |      9.6798e+13
第 14 层    |     -6.3032e+13 |      1.1983e+15
第 15 层    |     -1.4353e+13 |      1.2861e+16
第 16 层    |      8.2259e+15 |      1.4823e+17
第 17 层    |     -4.5056e+16 |      1.6880e+18
第 18 层    |     -1.2790e+18 |      1.8781e+19
第 19 层    |     -1.8206e+18 |      2.0794e+20
第 

{'final_mean': -0.0010135134216398, 'final_std': 0.14771540462970734}

## 4.9 环境和分布偏移 (Environment and Distribution Shift)

> 为什么在训练集上表现完美的模型，到了实际应用（生产环境）中会突然“翻车”？

### 1. 核心矛盾：IID 假设与现实世界

在机器学习的实验室环境下，我们通常假设数据是 **独立同分布 (IID)** 的：
*   **IID 假设**：训练数据和测试数据都是从同一个概率分布 $P(\mathbf{x}, y)$ 中独立采样得到的。
*   **现实挑战**：环境是动态的（非平稳）。训练集可能已经过时，或者采集环境与实际部署环境存在巨大差异。一旦 IID 假设失效，模型的泛化能力将迅速崩溃。

---

### 2. 数学根源：期望风险 vs. 经验风险

优化的目标与最终目标的鸿沟：

1.  **实际风险 (Expected Risk)**：模型在**全量真实分布** $P(\mathbf{x}, y)$ 上的误差期望。
    $$R[f] = E_{(\mathbf{x}, y) \sim P} [L(f(\mathbf{x}), y)] = \int \int L(f(\mathbf{x}), y) p(\mathbf{x}, y) d\mathbf{x} dy$$
2.  **经验风险 (Empirical Risk)**：模型在**有限训练集** ($n$ 个样本) 上的平均误差。
    $$\hat{R}_n[f] = \frac{1}{n} \sum_{i=1}^n L(f(\mathbf{x}_i), y_i)$$
3.  **结论**：我们只能最小化“经验风险”来逼近“实际风险”。当分布发生偏移时，经验风险的最小值不再是实际风险的最小值。

---

### 3. 分布偏移的深度分类 (Taxonomy)

| 偏移类型 | 数学定义 | 核心含义 | 应对可能性 |
| :--- | :--- | :--- | :--- |
| **协变量偏移 (Covariate Shift)** | $P(\mathbf{x})$ 改变，但 $P(y \mid \mathbf{x})$ 不变 | 特征分布变了，但判定规则没变。 | **可纠正**（通过加权） |
| **标签偏移 (Label Shift)** | $P(y)$ 改变，但 $P(\mathbf{x} \mid y)$ 不变 | 类别比例变了，但每一类的特征呈现没变。 | **可纠正**（通过加权） |
| **概念偏移 (Concept Shift)** | $P(y \mid \mathbf{x})$ 改变 | **判定规则变了**。同样的输入，昨天的标签和今天不同。 | **不可直接纠正**（需新数据） |

---

### 4. 分布偏移的数学纠正 (Importance Sampling)

对于前两类偏移，我们可以通过**重要性采样**在不重新训练模型架构的情况下进行修正。

#### 4.1 协变量偏移纠正
设训练分布为 $p(\mathbf{x})$，测试分布为 $q(\mathbf{x})$。我们需要为训练集中的每个样本赋予权重 $\beta_i$：
$$\beta_i = \frac{q(\mathbf{x}_i)}{p(\mathbf{x}_i)}$$
通过加权损失函数 $\sum \beta_i L(f(\mathbf{x}_i), y_i)$ 进行训练，可以使经验风险逼近测试集上的实际风险。

#### 4.2 为什么概念偏移无法“纠正”？
*   **数学逻辑**：概念偏移意味着映射关系 $f: x \to y$ 本身发生了断裂。
*   **工程结论**：历史数据中不包含新规则的任何信息。因此，概念偏移无法通过数学加权修复，只能通过**持续学习 (Continual Learning)** 或 **定期重训练** 来更新模型逻辑。

---

### 5. 机器学习任务的全景分类

D2L 强调了在不同环境下，数据流和反馈机制的差异：

1.  **批量学习 (Batch Learning)**：一次训练，长期部署。最怕环境偏移。
2.  **在线学习 (Online Learning)**：数据流式进入，模型实时更新。能较好应对渐进式偏移。
3.  **老虎机 (Bandits)**：模型必须在“探索（试错）”和“利用（获取最大收益）”之间权衡。
4.  **控制与强化学习**：环境是交互式的，模型的动作会改变未来的输入分布（如自动驾驶）。

---

### 6. 社会伦理与公平性

分布偏移不仅是技术问题，更是伦理问题：
*   **反馈循环 (Feedback Loops)**：模型预测 $\to$ 引导用户决策 $\to$ 产生新数据 $\to$ 强化模型偏见。
*   **覆盖偏移 (Coverage Shift)**：如果训练集完全没见过某类人群的数据，模型在该类人群上的失败将是必然的。

---

### 7. 关键工程暗知识 (Engineering Wisdom)

1.  **数据抽样陷阱**：
    在划分训练集和测试集时，如果你是**按时间**划分（前 10 个月训练，后 2 个月测试），这通常能比**随机划分**更早发现分布偏移问题。
2.  **非平稳环境**：
    在金融预测或自动驾驶中，环境是瞬息万变的。在这种情况下，**验证集的准确率比训练集的准确率重要 100 倍**。
3.  **鲁棒性验证**：
    在多个不同的数据集上测试，而不仅仅是一个固定的测试集。

---

### 8. 总结

**算法不是孤立的数学公式，而是复杂环境系统的一部分。**
*   **协变量/标签偏移**：靠数学解决（加权）。
*   **概念偏移**：靠工程解决（监控、重练、在线微调）。
*   **系统稳定性**：靠监控解决（对比训练误差与实时验证误差）。


In [4]:
# 重要性采样算子实现
import torch 
from torch import Tensor
from typing import Optional

def compute_importance_weights(
    train_probs: Tensor,
    test_probs: Tensor,
    clip_val: float
) -> Tensor:
    """计算并规范化重要性采样权重 beta = q / p。
    
    Args:
        train_probs: 样本在训练集分布下的概率密度。
        test_probs: 样本在测试集分布下的概率密度。
        clip_val: 权重裁剪阈值，防止过大权重导致梯度爆炸。

    Returns:
        Tensor: 规范化后的权重向量。
    """
    # 计算原始权重
    beta: Tensor = test_probs / train_probs

    # 执行裁剪，以确保数值稳定性
    beta = torch.clamp(beta, max=clip_val)

    # 归一化，保持 Batch 能量一致
    return beta / beta.mean()

In [5]:
# 分布偏移修正引擎
def train_epoch_with_weights(
    net: torch.nn.Module,
    train_iter: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    updater: torch.optim.Optimizer,
    weight_fn: Optional[Callable[[Tensor], Tensor]] = None
) -> float:
    """带权重的训练引擎，用以应对分布偏移。"""
    net.train()
    total_loss = 0.0
    sample_count = 0

    for X, y in train_iter:
        updater.zero_grad()
        # 1. 前向传播
        y_hat = net(X)
        l = loss_fn(y_hat, y)

        # 2. 如果存在偏移修正权重
        if weight_fn is not None:
            weights = weight_fn(X)
            l = l * weights

        # 3. 反向传播
        l.mean().backward()
        updater.step()

        total_loss += l.sum().item()
        sample_count += y.numel()

    return total_loss / sample_count

**“协变量偏移修正”对比实验**
*   **任务**：简单的二分类，$y = 1$ if $x > 0$ else $0$。
*   **训练集分布 $p(x)$**：均值为 -2 的正态分布（大部分数据在负数区，模型很少见到正数样本）。
*   **测试集分布 $q(x)$**：均值为 +2 的正态分布（大部分数据在正数区）。
*   **挑战**：普通训练会让模型在 $x > 0$ 区域表现很差，因为训练集里这类样本太少。


In [23]:
import torch
from torch import nn, Tensor
from torch.distributions import Normal
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class ImportanceSamplingConfig:
    lr: float = 0.1
    epochs: int = 100
    batch_size: int = 1000 # 使用大 Batch 方便观察分布

    # 该实验特有配置
    train_mean: float = -2.0
    test_mean: float = 2.0

def compute_importance_weights(
    x: Tensor,
    p_dist: Normal,
    q_dist: Normal,
    clip_val: float = 10.0
) -> Tensor:
    """计算并规范化重要性采样权重 beta = q(x) / p(x)。"""
    # 1. 计算在两个分布下的概率密度 (PDF)
    # log_prob 返回的是对数概率，exp 后得到原始概率密度
    p_probs = p_dist.log_prob(x).exp()
    q_probs = q_dist.log_prob(x).exp()

    # 2. 计算 beta = q / p (加 1e-8 防止除零错误)
    beta = q_probs / (p_probs + 1e-8)
    
    # 3. 裁剪与规范化
    beta = torch.clamp(beta, max=clip_val)
    return beta / beta.mean()

> *   **`nn.Sigmoid()`**：
>       *   在二分类中，我们不使用 Softmax，而是用 Sigmoid。输出的那个数字直接代表 **“属于类别 1 的概率”**。

> *   **`nn.BCELoss` (Binary Cross Entropy Loss)**：
>       *   **用途**：专门用于二分类的交叉熵损失函数。
>       *   **输入要求**：它要求输入的 `y_hat` 必须是经过 Sigmoid 处理后的概率值（0到1之间）。

In [ ]:
def run_importance_experiment(config: ImportanceSamplingConfig):
    # 1. 数据准备
    # 定义分布对象
    p_dist = Normal(config.train_mean, 1.0)
    q_dist = Normal(config.test_mean, 1.0)

    # 生成训练数据 (来自 p) 和 测试数据 (来自 q)
    train_x = p_dist.sample((config.batch_size, 1))
    train_y = (train_x > 0).float()

    test_x = q_dist.sample((config.batch_size, 1))
    test_y = (test_x > 0).float()

    # 2. 计算重要性权重
    # 这一步让模型知道：训练集中那些靠近 0 或大于 0 的稀有样本，在测试集中其实很重要
    weights = compute_importance_weights(train_x, p_dist, q_dist)

    # 3. 定义模型 和 训练循环逻辑
    def train_model(use_weights: bool) -> nn.Module:
        model = nn.Sequential(nn.Linear(1, 1), nn.Sigmoid())
        optimizer = torch.optim.SGD(model.parameters(), lr=config.lr)
        loss_fn = nn.BCELoss(reduction="none") # 使用 "none" 手动加权 

        for _ in range(config.epochs):
            optimizer.zero_grad()
            y_hat = model(train_x)
            l = loss_fn(y_hat, train_y)

            if use_weights:
                l *= weights

            l.mean().backward()
            optimizer.step()
        return model
    
    # 4. 测试逻辑
    def evaluate(model: nn.Module, x: Tensor, label: Tensor, name: str) -> None:
        with torch.no_grad():
            # Sigmoid() 将范围缩为 (0, 1)
            preds = (model(x) > 0.5).float()
            acc = (preds == label).float().mean()
            print(f"{name} 在测试集上的准确率: {acc:.4f}")
    
    # 5. 结果对比
    model_standard = train_model(use_weights=False)
    model_weighted = train_model(use_weights=True)

    print(f"训练分布均值: {config.train_mean}, 测试分布均值: {config.test_mean}\n")
    evaluate(model_standard, test_x, test_y, "普通训练 (ERM)")
    evaluate(model_weighted, test_x, test_y, "重要性采样训练 (Corrected)")

# 由于模型本身太简单了，下面那个准确率可能会出现一些很奇怪的东西。
run_importance_experiment(ImportanceSamplingConfig())

训练分布均值: -2.0, 测试分布均值: 2.0

普通训练 (ERM) 在测试集上的准确率: 0.8070
重要性采样训练 (Corrected) 在测试集上的准确率: 0.9770


**重要性采样的局限**：
1.  **概率密度的不可知性**：
    在实验中，我们作为“上帝”知道 $p(x)$ 和 $q(x)$ 的数学公式。但在现实中，我们只有两堆数据。
    *   **进阶做法**：通常会先训练一个**判别器（Discriminator）**，用来区分样本是来自训练集还是测试集。判别器的输出可以转化为 $\beta$。
2.  **支持度（Support）问题**：
    如果训练集里**完全没有** $x > 0$ 的样本，无论你给权重乘多少倍，模型也学不会正数的规律。重要性采样只能修正“比例失调”，不能无中生有。